In [8]:
# Camada Silver 
# Regras aplicadas explicitamente
# Métricas de qualidade
# Visualização final para validação

import sqlite3
from datetime import date
import pandas as pd


In [2]:
conexao  = sqlite3.connect(database='db_project_eng_dados')

In [7]:
# Criar a tabela Silver a partir da Bronze:

conexao.execute("""
CREATE TABLE IF NOT EXISTS silver_produtos AS SELECT DISTINCT
    id_produto,
    nome_produto,
    categoria,
    tipo_dado,
    resolucao_espacial,
    sistema_coordenadas,
    area_cobertura_km2,
    date(data_aquisicao) AS data_aquisicao,
    formato,
    fornecedor,
    preco,
    date(data_bronze) AS data_bronze,
    date ('now') AS data_silver
  
FROM bronze_produtos""")

conexao.commit()

# deduplicação (DISTINCT)
# padronização de datas com nova coluna de linhagem (data_silver)


In [9]:
# Visualização amostral da Camada Silver:

df_silver_check = pd.read_sql ("""
SELECT * FROM silver_produtos
LIMIT 10 """, conexao)

df_silver_check

,id_produto,nome_produto,categoria,tipo_dado,resolucao_espacial,sistema_coordenadas,area_cobertura_km2,data_aquisicao,formato,fornecedor,preco,data_bronze,data_silver
0,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,10m,SIRGAS 2000 / UTM 23S,1500.0,2023-06-15,Shapefile,GeoMapas Ltda,3500.0,2025-12-22,2025-12-23
1,2,Modelo Digital de Elevação,MDE,Raster,30m,WGS84,5000.0,2022-11-20,GeoTIFF,INPE,0.0,2025-12-22,2025-12-23
2,3,Ortoimagem Urbana São Paulo,Imagem Orbital,Raster,0.5m,SIRGAS 2000,800.0,2023-02-10,GeoTIFF,Maxar,12000.0,2025-12-22,2025-12-23
3,4,Mapa de Drenagem Hidrográfica,Hidrografia,Vetorial,1:25000,SIRGAS 2000,3200.0,2021-08-05,GeoPackage,ANA,0.0,2025-12-22,2025-12-23
4,5,Classificação de Vegetação Cerrado,Vegetação,Raster,20m,WGS84,2100.0,2022-09-30,GeoTIFF,IBGE,2500.0,2025-12-22,2025-12-23
5,6,Limites Administrativos Municipais,Base Cartográfica,Vetorial,1:50000,SIRGAS 2000,8500.0,2023-01-01,Shapefile,IBGE,0.0,2025-12-22,2025-12-23
6,7,Mapa de Risco de Deslizamento,Análise Ambiental,Vetorial,1:10000,SIRGAS 2000 / UTM 22S,600.0,2023-07-12,GeoPackage,Defesa Civil,4800.0,2025-12-22,2025-12-23


In [10]:
# Métricas de qualidade:

df_metricas_silver = pd.read_sql ("""
SELECT 
    COUNT (*) AS total_registros,
    COUNT (DISTINCT id_produto) AS produtos_unicos,
    SUM (preco IS NULL) AS preco_nulo,
    MIN (data_silver) AS data_processamento
FROM silver_produtos """, conexao)

df_metricas_silver




,total_registros,produtos_unicos,preco_nulo,data_processamento
0,7,7,0,2025-12-23


In [12]:
# Visualização total da tabela SILVER:

df_table_silver = pd.read_sql ("""
SELECT * FROM silver_produtos """, conexao)

df_table_silver

,id_produto,nome_produto,categoria,tipo_dado,resolucao_espacial,sistema_coordenadas,area_cobertura_km2,data_aquisicao,formato,fornecedor,preco,data_bronze,data_silver
0,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,10m,SIRGAS 2000 / UTM 23S,1500.0,2023-06-15,Shapefile,GeoMapas Ltda,3500.0,2025-12-22,2025-12-23
1,2,Modelo Digital de Elevação,MDE,Raster,30m,WGS84,5000.0,2022-11-20,GeoTIFF,INPE,0.0,2025-12-22,2025-12-23
2,3,Ortoimagem Urbana São Paulo,Imagem Orbital,Raster,0.5m,SIRGAS 2000,800.0,2023-02-10,GeoTIFF,Maxar,12000.0,2025-12-22,2025-12-23
3,4,Mapa de Drenagem Hidrográfica,Hidrografia,Vetorial,1:25000,SIRGAS 2000,3200.0,2021-08-05,GeoPackage,ANA,0.0,2025-12-22,2025-12-23
4,5,Classificação de Vegetação Cerrado,Vegetação,Raster,20m,WGS84,2100.0,2022-09-30,GeoTIFF,IBGE,2500.0,2025-12-22,2025-12-23
5,6,Limites Administrativos Municipais,Base Cartográfica,Vetorial,1:50000,SIRGAS 2000,8500.0,2023-01-01,Shapefile,IBGE,0.0,2025-12-22,2025-12-23
6,7,Mapa de Risco de Deslizamento,Análise Ambiental,Vetorial,1:10000,SIRGAS 2000 / UTM 22S,600.0,2023-07-12,GeoPackage,Defesa Civil,4800.0,2025-12-22,2025-12-23


In [ ]:
df_table_silver.dtypes

# Isso nos mostra que 'object' é formato TXT. O pandas vê isso string como objeto.

id_produto              object
nome_produto            object
categoria               object
tipo_dado               object
resolucao_espacial      object
sistema_coordenadas     object
area_cobertura_km2     float64
data_aquisicao          object
formato                 object
fornecedor              object
preco                  float64
data_bronze             object
data_silver             object
dtype: object

In [ ]:
# Com esse comando, podemos ver o schema real da tabela SILVER ou de qualquer tabela.

df_schema = pd.read_sql("""
PRAGMA table_info(silver_produtos)
""", conexao)

df_schema


,cid,name,type,notnull,dflt_value,pk
0,0,id_produto,TEXT,0,None,0
1,1,nome_produto,TEXT,0,None,0
2,2,categoria,TEXT,0,None,0
3,3,tipo_dado,TEXT,0,None,0
4,4,resolucao_espacial,REAL,0,None,0
5,5,sistema_coordenadas,TEXT,0,None,0
6,6,area_cobertura_km2,REAL,0,None,0
7,7,data_aquisicao,,0,None,0
8,8,formato,TEXT,0,None,0
9,9,fornecedor,TEXT,0,None,0
